In [1]:
import os
import shutil
!pip install uv
!git clone https://github.com/Uxvan/assignment1-basics.git
os.chdir('/content/assignment1-basics')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.0/27.0 MB 66.4 MB/s eta 0:00:00
Cloning into 'assignment1-basics'...
remote: Enumerating objects: 940, done.
remote: Counting objects: 100% (243/243), done.
remote: Compressing objects: 100% (99/99), done.
remote: Total 940 (delta 226), reused 144 (delta 144), pack-reused 697 (from 4)
Receiving objects: 100% (940/940), 19.68 MiB | 6.88 MiB/s, done.
Resolving deltas: 100% (590/590), done.


In [ ]:
os.cpu_count()

2

In [2]:
!mkdir -p data
!cd data

!wget https://huggingface.co/datasets/roneneldan/TinyStories/resolve/main/TinyStoriesV2-GPT4-train.txt
!wget https://huggingface.co/datasets/roneneldan/TinyStories/resolve/main/TinyStoriesV2-GPT4-valid.txt

!cd ..

--2026-07-26 15:03:40--  https://huggingface.co/datasets/roneneldan/TinyStories/resolve/main/TinyStoriesV2-GPT4-train.txt
Resolving huggingface.co (huggingface.co)... 13.35.202.121, 13.35.202.40, 13.35.202.34, ...
Connecting to huggingface.co (huggingface.co)|13.35.202.121|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://us.aws.cdn.hf.co/xet-bridge-us/645e8da96320b0efe40ade7a/02e40cc51c59a4bc6c51bd7bc9acda4316e208745be060558eaf500cd14e9f96?response-content-type=text%2Fplain&user_id=public&X-Xet-Cas-Uid=public&response-content-disposition=inline%3B+filename*%3DUTF-8%27%27TinyStoriesV2-GPT4-train.txt%3B+filename%3D%22TinyStoriesV2-GPT4-train.txt%22%3B&Expires=1785081820&Policy=eyJTdGF0ZW1lbnQiOlt7IlJlc291cmNlIjoiaHR0cHM6Ly91cy5hd3MuY2RuLmhmLmNvL3hldC1icmlkZ2UtdXMvNjQ1ZThkYTk2MzIwYjBlZmU0MGFkZTdhLzAyZTQwY2M1MWM1OWE0YmM2YzUxYmQ3YmM5YWNkYTQzMTZlMjA4NzQ1YmUwNjA1NThlYWY1MDBjZDE0ZTlmOTZcXD9yZXNwb25zZS1jb250ZW50LXR5cGU9dGV4dCUyRnBsYWluJnVzZXJfaWQ9cHVibGljJl

In [3]:
import numpy as np
from multiprocessing import Pool
from cs336_basics.tokenizer import Tokenizer

VOCAB_PATH = 'train_results/vocab_merges/vocab_tinystories.pkl'
MERGES_PATH = 'train_results/vocab_merges/merges_tinystories.pkl'

# 每个worker进程会各自初始化一份tokenizer(避免跨进程共享大对象的序列化开销)
_worker_tokenizer = None

def init_worker():
    global _worker_tokenizer
    _worker_tokenizer = Tokenizer.from_files(VOCAB_PATH, MERGES_PATH, special_tokens=['<|endoftext|>'])

def encode_lines(lines):
    ids = []
    for line in lines:
        ids.extend(_worker_tokenizer.encode(line))
    return ids

def tokenize_and_save_parallel(input_txt_path, output_bin_path, num_workers=2, lines_per_chunk=10000):
    def line_chunks():
        buf = []
        with open(input_txt_path, 'r') as f:
            for line in f:
                buf.append(line)
                if len(buf) >= lines_per_chunk:
                    yield buf
                    buf = []
        if buf:
            yield buf

    total_tokens = 0
    with Pool(num_workers, initializer=init_worker) as pool:
        with open(output_bin_path, 'wb') as out_f:
            # imap 保持输入顺序输出,chunksize=1 因为每个任务本身已经是一批行
            for ids in pool.imap(encode_lines, line_chunks(), chunksize=1):
                arr = np.array(ids, dtype=np.uint16)
                arr.tofile(out_f)
                total_tokens += len(ids)
                print(f"已处理 {total_tokens:,} tokens")

    print(f"完成,共 {total_tokens:,} tokens,保存到 {output_bin_path}")

tokenize_and_save_parallel('TinyStoriesV2-GPT4-train.txt', 'data/tinystories_train.bin', num_workers=2)
tokenize_and_save_parallel('TinyStoriesV2-GPT4-valid.txt', 'data/tinystories_valid.bin', num_workers=2)

/content/assignment1-basics/cs336_basics/tokenizer.py:5: SyntaxWarning: invalid escape sequence '\p'
  PAT = r"""'(?:[sdmt]|ll|ve|re)| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+"""


已处理 346,488 tokens
已处理 688,558 tokens
已处理 1,030,865 tokens
已处理 1,378,052 tokens
已处理 1,722,559 tokens
已处理 2,068,958 tokens
已处理 2,416,153 tokens
已处理 2,760,534 tokens
已处理 3,107,048 tokens
已处理 3,454,850 tokens
已处理 3,802,562 tokens
已处理 4,147,334 tokens
已处理 4,490,316 tokens
已处理 4,836,212 tokens
已处理 5,183,452 tokens
已处理 5,528,548 tokens
已处理 5,874,220 tokens
已处理 6,222,027 tokens
已处理 6,572,591 tokens
已处理 6,919,389 tokens
已处理 7,269,216 tokens
已处理 7,615,682 tokens
已处理 7,955,703 tokens
已处理 8,304,498 tokens
已处理 8,649,399 tokens
已处理 8,997,974 tokens
已处理 9,341,304 tokens
已处理 9,686,863 tokens
已处理 10,033,769 tokens
已处理 10,381,864 tokens
已处理 10,731,306 tokens
已处理 11,073,518 tokens
已处理 11,422,710 tokens
已处理 11,769,061 tokens
已处理 12,117,615 tokens
已处理 12,469,251 tokens
已处理 12,811,290 tokens
已处理 13,160,116 tokens
已处理 13,504,744 tokens
已处理 13,857,576 tokens
已处理 14,201,915 tokens
已处理 14,551,641 tokens
已处理 14,896,094 tokens
已处理 15,240,887 tokens
已处理 15,589,736 tokens
已处理 15,940,509 tokens
已处理 16,286,468 token

In [2]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [4]:
import shutil
shutil.copy('/content/drive/MyDrive/tinystories_valid.bin', '/content/assignment1-basics/tinystories_valid.bin')

shutil.copy('/content/drive/MyDrive/tinystories_train.bin', '/content/assignment1-basics/tinystories_train.bin')



'/content/assignment1-basics/tinystories_valid.bin'

In [3]:
os.makedirs('/content/assignment1-basics/checkpoints', exist_ok=True)

In [10]:
!mv /content/trainingScript.py /content/assignment1-basics/cs336_basics/trainingScript.py

In [12]:
%cd /content/assignment1-basics
!python -m cs336_basics.trainingScript \
--train_data /content/assignment1-basics/tinystories_train.bin \
--val_data /content/assignment1-basics/tinystories_valid.bin \
--checkpoint_path checkpoints/tinystories_ckpt.pt

/content
iter 0: train loss 9.2661, lr 0.000000, elapsed 0.9s
iter 0: val loss 9.2633
iter 10: train loss 9.1314, lr 0.000050, elapsed 12.7s
iter 20: train loss 8.6239, lr 0.000100, elapsed 17.0s
iter 30: train loss 7.6272, lr 0.000150, elapsed 21.3s
iter 40: train loss 6.6583, lr 0.000200, elapsed 25.5s
iter 50: train loss 5.7876, lr 0.000250, elapsed 29.8s
iter 60: train loss 5.2214, lr 0.000300, elapsed 34.1s
iter 70: train loss 4.8109, lr 0.000350, elapsed 38.4s
iter 80: train loss 4.4117, lr 0.000400, elapsed 42.7s
iter 90: train loss 4.2376, lr 0.000450, elapsed 47.0s
iter 100: train loss 4.0337, lr 0.000500, elapsed 51.4s
iter 110: train loss 3.9246, lr 0.000550, elapsed 55.7s
iter 120: train loss 3.8133, lr 0.000600, elapsed 60.1s
iter 130: train loss 3.6050, lr 0.000650, elapsed 64.5s
iter 140: train loss 3.6026, lr 0.000700, elapsed 68.9s
iter 150: train loss 3.3540, lr 0.000750, elapsed 73.3s
iter 160: train loss 3.4701, lr 0.000800, elapsed 77.7s
iter 170: train loss 3.4111

In [16]:
shutil.copy('/content/assignment1-basics/checkpoints/tinystories_ckpt.pt', '/content/drive/MyDrive/tinystories_ckpt.pt')

'/content/drive/MyDrive/tinystories_ckpt.pt'

In [4]:
import shutil

In [5]:
shutil.copy('/content/drive/MyDrive/tinystories_ckpt.pt', '/content/assignment1-basics/checkpoints/tinystories_ckpt.pt')

'/content/assignment1-basics/checkpoints/tinystories_ckpt.pt'

In [6]:
%cd /content/assignment1-basics
!python -m cs336_basics.inference \
--prompt 'I am' \
--checkpoint_path /content/assignment1-basics/checkpoints/tinystories_ckpt.pt

/content
/content/assignment1-basics/cs336_basics/tokenizer.py:5: SyntaxWarning: invalid escape sequence '\p'
  PAT = r"""'(?:[sdmt]|ll|ve|re)| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+"""
I am Tom the dog, and I love animals. Let's play together!" Lily smiled and said, "Yes, let's play!"
As they played, Lily's mom came into the room. She saw the mouse and said, "Hi, Lily! What's your name?" Lily said, "His name is Fluffy. Nice to meet Fluffy and we can all play together!" Tom and Lily played all day and had lots of fun. They became best friends, and the room was a happy place
